# Avance 5 - Modelo final de la permeabilidad

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Combina modelos para clasificar la permeabilidad de la barrera hematoencefálica (BBB+ / BBB-), la fase 1 del proyecto. Parte de los modelos individuales del [Avance 4](04_Models.ipynb), arma cinco ensambles que cubren las dos estrategias -**homogéneos**, que combinan copias de una misma familia, y **heterogéneos**, que combinan familias distintas-, los compara con el mejor individual en la misma corrida y elige el modelo final. Al final, y una sola vez, lo evalúa en la partición de prueba que ningún cuaderno anterior tocó.

La fase 2, la potencia contra cada diana, va en el 05b.

Usa los mismos datos, la misma partición agrupada por andamio y esqueleto de conectividad, las mismas tres semillas y los mismos cinco pliegues que el 03 y el 04, así que sus cifras se comparan con las de esos cuadernos. Entra **solo con descriptores**: el [03c](03c_Permeabilidad_huellas.ipynb) midió que las huellas Sort & Slice no mejoran a ningún modelo en esta fase.

## 1. Preparación

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import json
import platform
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"
DATOS = RAIZ / "notebooks" / "datos"

from src.evaluation.cv import _medir, evaluar, intervalo_por_grupos, tabla_comparativa
from src.evaluation.graficos import (curva_precision_exhaustividad, curva_roc,
                                     importancias_dos_metodos, matriz_confusion)
from src.models import cargar, seleccion
from src.models.ajuste import BusquedaAnidada
from src.models.ensambles_clasificacion import ensambles
from src.tracking import tomar_prueba

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  macOS-27.0.1-arm64-arm-64bit
python   3.12.12
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### Los datos

Entrenamiento y validación de la fase 1, con la prueba reservada fuera. Cada compuesto lleva su **origen** -natural exacto, análogo estereoquímico o sin equivalente natural-, que es el desglose de la decisión [1.6](../docs/decisiones.md#d1-6): un promedio de todos no dice cómo le va al modelo con los productos naturales, que son los que el proyecto quiere priorizar.

In [2]:
t = cargar("permeabilidad", estratos="origen")
X, y, g, origen = t.X, t.y, t.grupos, t.estratos
print(f"{len(y):,} compuestos · {X.shape[1]} descriptores · {y.mean():.1%} BBB+ · "
      f"{len(np.unique(g)):,} grupos")
print(pd.Series(origen).value_counts().to_string())

6,306 compuestos · 151 descriptores · 65.0% BBB+ · 1,048 grupos
natural exacto             2920
sin equivalente natural    2618
análogo estereoquímico      768


## 2. Los modelos

### Los individuales de referencia

Los cuatro mejores del Avance 4 por MCC, sin afinar: el SVM con núcleo RBF, la regresión logística, la red de una capa y los vecinos más cercanos. El SVM fue el individual que eligió el 04 y es **el mejor individual** contra el que se mide cada ensamble.

**Se vuelven a correr aquí, y las cifras del 04 no se usan como referencia.** La comparación tiene que salir de una sola corrida -mismos pliegues, mismas semillas y misma máquina- para que la columna de tiempos signifique lo mismo en todas las filas. Y hay una razón más fuerte: las cifras registradas del 04 (el SVM en 0.703 de MCC) se midieron sobre otra matriz de descriptores. La misma versión de RDKit calcula flotantes distintos según cómo se compiló, así que la matriz de la fase 1 no es idéntica entre las máquinas del equipo ([#166](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/166)), y cuál se declara canónica está por decidir. Lo que no depende de esa decisión es la comparación de aquí: **todos los modelos de esta corrida, individuales y ensambles, leen la misma matriz**, y la diferencia pareada contra el SVM medido aquí es limpia aunque la matriz cambie después. `tracking.py` anota el sha256 de lo que leyó, así que siempre se sabrá con cuál se midió.

Esos cuatro son también la base de la votación y del stacking, como pide la rúbrica.

### Los ensambles

| Ensamble | Estrategia | Que combina |
|:--|:--|:--|
| `bosque` | homogénea, bagging de árboles con columnas al azar | 300 árboles de decisión |
| `boosting` | homogénea, secuencial: cada árbol corrige lo que falló el anterior | árboles poco profundos |
| `bagging_svm` | homogénea, bagging | 10 copias del SVM RBF, cada una sobre un remuestreo de las filas |
| `votacion` | heterogénea, promedio de las probabilidades | los cuatro individuales |
| `stacking` | heterogénea, una regresión logística aprende a combinarlos | los cuatro individuales |

El bosque y el boosting son los del baseline del 03, sin cambios, para que sus filas se puedan contrastar con lo que ya está registrado; además van sus dos versiones afinadas.

**El stacking es el único que puede filtrar sin que se note.** Su meta-modelo aprende de predicciones que los modelos base hacen fuera de pliegue, y esos pliegues internos se arman al azar por omisión: con análogos del mismo esqueleto a los dos lados, las predicciones de entrenamiento salen mejores de lo que serán en validación y el meta-modelo aprende a confiar de más en el que memoriza. El ajuste externo seguiría viéndose honesto. `StackingAgrupado` parte con los grupos del pliegue externo, por la misma razón que `BusquedaAnidada` ([1.2](../docs/decisiones.md#d1-2)).

**Lo que se ejecuta es el código de `src/`**, que importa la celda de arriba. Se muestra aquí para leerlo sin abrir otro archivo, y `tests/test_05_codigo.py` comprueba que lo mostrado y lo importado sean idénticos.

```python
class StackingAgrupado(BaseEstimator, ClassifierMixin):
    """Stacking cuya validación interna agrupa con los grupos del pliegue externo.

    `estimadores` es una lista de `(nombre, canalización)`. El meta-modelo recibe la
    probabilidad de la clase positiva de cada uno, calculada fuera de los pliegues
    internos, y por defecto es una regresión logística: con tantas entradas como
    modelos base, y correlacionadas, un meta-modelo flexible sobreajusta.
    """

    necesita_grupos = True

    def __init__(self, estimadores: Sequence[tuple[str, BaseEstimator]],
                 final: BaseEstimator | None = None, *, pliegues: int = 3,
                 semilla: int = 0):
        self.estimadores = estimadores
        self.final = final
        self.pliegues = pliegues
        self.semilla = semilla

    def fit(self, X, y, grupos=None):
        if grupos is None:
            raise ValueError(
                "StackingAgrupado necesita `grupos`: sin ellos el meta-modelo aprendería "
                "de predicciones hechas con análogos del mismo esqueleto a los dos lados. "
                "El arnés los pasa solo; fuera de él, llame fit(X, y, grupos=...).")
        y = np.asarray(y)
        interno = StratifiedGroupKFold(n_splits=self.pliegues, shuffle=True,
                                       random_state=self.semilla)
        particiones = list(interno.split(X, y, np.asarray(grupos)))
        final = LogisticRegression(max_iter=3000) if self.final is None else clone(self.final)
        self.stacking_ = StackingClassifier(
            [(n, clone(e)) for n, e in self.estimadores], final_estimator=final,
            cv=particiones, stack_method="predict_proba").fit(X, y)
        self.classes_ = self.stacking_.classes_
        return self

    def predict(self, X):
        return self.stacking_.predict(X)

    def predict_proba(self, X):
        return self.stacking_.predict_proba(X)

def ensambles(base: Sequence[str], semilla: int = 0) -> dict[str, BaseEstimator]:
    """Los cinco ensambles, con `base` como los individuales de la votación y el stacking.

    `base` son claves del catálogo de `alternativos.py`; el bagging copia el SVM RBF
    porque es el mejor individual del Avance 4. El bosque y el boosting son los del
    baseline del 03, sin cambios, para que su fila se pueda contrastar con la que ya
    está registrada.
    """
    individuales = seleccion(tuple(base), semilla)
    referencia = canalizaciones(semilla)
    # El bosque del baseline usa todos los núcleos; aquí uno solo, como los demás, para que
    # la columna de tiempos compare modelos y no paralelismo. No cambia sus predicciones.
    referencia["bosque"].set_params(modelo__n_jobs=1)
    svm = seleccion(("svm_rbf",), semilla)["svm_rbf"]
    return {
        "bosque": referencia["bosque"],
        "boosting": referencia["boosting"],
        "bagging_svm": Pipeline([
            ("modelo", BaggingClassifier(svm, n_estimators=10, random_state=semilla))]),
        "votacion": Pipeline([
            ("modelo", VotingClassifier(list(individuales.items()), voting="soft"))]),
        "stacking": StackingAgrupado(list(individuales.items()), semilla=semilla),
    }
```

### El ajuste de hiperparámetros

La rúbrica pide optimizar los modelos relevantes. Se afinan el bosque y el boosting, que son los dos ensambles homogéneos con hiperparámetros que cambian su comportamiento; el bagging del SVM hereda los del individual que ya eligió el 04, y la votación y el stacking no tienen propios más allá de sus bases.

Las dos rejillas tienen 18 combinaciones y se recorren completas, con tres pliegues internos **también agrupados**: la búsqueda vive dentro de cada pliegue externo, así que la validación del arnés nunca participa en elegir. La métrica que elige es el MCC, la principal de la fase 1 ([C.1](../docs/decisiones.md#dc-1)). La columna `elegidos` del resultado guarda qué eligió en cada pliegue, que es lo que permite decir si la elección fue estable o cambio con la partición.

| Modelo | Parámetro | Valores | Qué controla |
|:--|:--|:--|:--|
| `bosque` | `max_features` | `sqrt`, 0.3, 0.5 | cuántas columnas ve cada división, que es lo que diversifica los árboles |
| | `min_samples_leaf` | 1, 2, 5 | cuánto puede memorizar cada hoja |
| | `max_depth` | sin límite, 20 | un freno adicional al sobreajuste |
| `boosting` | `learning_rate` | 0.03, 0.1, 0.3 | cuánto corrige cada árbol |
| | `max_leaf_nodes` | 15, 31, 63 | la complejidad de cada arbol |
| | `l2_regularization` | 0, 1 | la penalización de las hojas |

**No se usa la parada temprana del boosting**, que separa su propia validación al azar y sin agrupar: volvería a meter la fuga que todo este arnés evita.

In [3]:
SEMILLAS = (0, 1, 2)
BASE = ("svm_rbf", "logistica", "mlp", "knn")     # los cuatro mejores del 04 por MCC
MEJOR_INDIVIDUAL = "svm_rbf"                     # el que eligió el 04; contra el se mide cada ensamble
MINIMO_PLIEGUES = 12                             # de 15 evaluaciones, en cuántas tiene que mejorar
PRINCIPALES = ("mcc", "auprc_minoritaria")       # las que deciden; el AUROC es tercera lectura
MUESTRA_ENTRENAMIENTO = 3000                     # filas de entrenamiento para medir la brecha

REJILLAS = {
    "bosque": {"modelo__max_features": ["sqrt", 0.3, 0.5],
               "modelo__min_samples_leaf": [1, 2, 5],
               "modelo__max_depth": [None, 20]},
    "boosting": {"modelo__learning_rate": [0.03, 0.1, 0.3],
                 "modelo__max_leaf_nodes": [15, 31, 63],
                 "modelo__l2_regularization": [0.0, 1.0]},
}

individuales = seleccion(BASE)
MODELOS = {**individuales, **ensambles(BASE)}
MODELOS.update({f"{n} afinado": BusquedaAnidada(MODELOS[n], r, pliegues=3)
                for n, r in REJILLAS.items()})
ORDEN = list(MODELOS)
ENSAMBLES = [n for n in ORDEN if n not in individuales]
print(f"{len(individuales)} individuales y {len(ENSAMBLES)} ensambles")
print(" · ".join(ENSAMBLES))

4 individuales y 7 ensambles
bosque · boosting · bagging_svm · votacion · stacking · bosque afinado · boosting afinado


### Lo que se fija antes de ver los resultados

- **Las métricas principales son el MCC y el AUPRC de la clase minoritaria** ([C.1](../docs/decisiones.md#dc-1)); el AUROC se reporta como tercera lectura y no decide. La tabla se ordena por MCC.
- **Un ensamble mejora al mejor individual solo si** la diferencia media supera el umbral **y** es positiva en al menos **12 de las 15** evaluaciones, en las dos métricas principales a la vez. Las diferencias se calculan contra `svm_rbf` **en la misma evaluación** (misma semilla, mismo pliegue), no entre promedios. Es la misma regla con la que se leyeron el 04c y el 03c ([1.3](../docs/decisiones.md#d1-3)), y se usa aquí por consistencia.
- **El umbral de cada métrica es la mayor desviación de esa métrica entre las 15 evaluaciones del mejor individual**, calculado con ese modelo solo, antes de mirar los ensambles. La regla **no está pre-registrada**: se fijó después de ver el control de los cuadernos anteriores. En la fase 1 el MCC varía mucho entre pliegues, así que el umbral sale alto y la regla es conservadora; si un ensamble gana en 15 de 15 con una diferencia estable por debajo del umbral, se reporta así y el umbral no se baja.
- **Los 15 pares no son independientes**: las semillas comparten filas y los pliegues parten los mismos datos. Los 12 de 15 son una prueba de signos con esa limitación, no una probabilidad.
- **Ordenar la tabla no declara un ganador.** La desviación entre evaluaciones es dispersión y no error estándar. Para afirmar que dos modelos se distinguen se remuestrean **grupos** con `intervalo_por_grupos`. El Avance 3 cerró con que ninguno de sus tres modelos se distinguía, y si eso se repite aquí, decirlo es el resultado.
- **Los desgloses por origen son exploratorios.** La comparación que decide es la del agregado.
- **La partición de prueba se toca una sola vez**, al final, con el modelo final ya elegido, a través de `tomar_prueba(final=True, motivo=...)`, que deja registro en `experimentos/prueba.csv` ([#54](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/54)).

## 3. La corrida

In [4]:
inicio = time.perf_counter()
res, fdp = evaluar(MODELOS, X, y, g, estratos=origen, semillas=SEMILLAS, verboso=False)
print(f"{len(res):,} filas · {(time.perf_counter() - inicio) / 60:.1f} min")

660 filas · 94.2 min


### La tabla comparativa

Una fila por modelo, ordenada por MCC, con el AUPRC de la minoritaria, el AUROC y el tiempo medio de ajuste por pliegue. `desv` es la dispersión del MCC entre las 15 evaluaciones, no un error estándar.

In [5]:
tabla = tabla_comparativa(res, principal="mcc")
tabla

,mcc,desv,auprc_minoritaria,auroc,segundos_ajuste,pliegues
modelo,,,,,,
stacking,0.7109,0.0698,0.8935,0.9318,9.1236,15
votacion,0.7092,0.0718,0.8949,0.9329,3.2754,15
bosque afinado,0.7023,0.0720,0.8936,0.9326,309.1378,15
boosting,0.7019,0.0633,0.8843,0.9302,0.7100,15
svm_rbf,0.7017,0.0731,0.8842,0.9190,1.4490,15
boosting afinado,0.6998,0.0632,0.8856,0.9293,32.7538,15
bosque,0.6996,0.0710,0.8950,0.9329,2.3971,15
bagging_svm,0.6948,0.0730,0.8862,0.9213,14.7252,15
logistica,0.6866,0.0663,0.8724,0.9133,0.0078,15


### Qué eligió la búsqueda en cada pliegue

Si los hiperparámetros elegidos cambian de un pliegue a otro, el afinado no está encontrando un óptimo estable sino acomodándose a cada partición, y esa es la lectura que importa, no la cifra.

In [6]:
elegidos = (res[res["elegidos"].notna() & res["estrato"].eq("todos")]
            .groupby("modelo")["elegidos"].value_counts().rename("evaluaciones"))
elegidos.to_frame()

evaluaciones
modelo           elegidos                                                        
boosting afinado {'modelo__l2_regularization': 0.0, 'modelo__lea...             6
                 {'modelo__l2_regularization': 0.0, 'modelo__lea...             3
                 {'modelo__l2_regularization': 1.0, 'modelo__lea...             2
                 {'modelo__l2_regularization': 1.0, 'modelo__lea...             1
                 {'modelo__l2_regularization': 0.0, 'modelo__lea...             1
                 {'modelo__l2_regularization': 1.0, 'modelo__lea...             1
                 {'modelo__l2_regularization': 0.0, 'modelo__lea...             1
bosque afinado   {'modelo__max_depth': None, 'modelo__max_featur...             6
                 {'modelo__max_depth': 20, 'modelo__max_features...             4
                 {'modelo__max_depth': 20, 'modelo__max_features...             1
                 {'modelo__max_depth': 20, 'modelo__max_features...             1
                 {'modelo__max_depth': None, 'modelo__max_featur...             1
                 {'modelo__max_depth': 20, 'modelo__max_features...             1
                 {'modelo__max_depth': None, 'modelo__max_featur...             1

### Los ensambles contra el mejor individual

Para cada ensamble se resta el MCC y el AUPRC del `svm_rbf` en **la misma evaluación**. `mejor_en` cuenta en cuántas de las 15 gana, y `mejora` aplica la regla fijada arriba. Los umbrales se calculan primero, solo con el mejor individual.

In [7]:
solo = res[res["estrato"].eq("todos")]
UMBRALES = {m: round(float(solo[solo["modelo"].eq(MEJOR_INDIVIDUAL)][m].std()), 3)
            for m in PRINCIPALES}
print(f"Umbrales, la desviacion de {MEJOR_INDIVIDUAL} entre sus 15 evaluaciones: {UMBRALES}")


def comparacion(r, estrato="todos"):
    """Diferencia de cada modelo contra el mejor individual, evaluacion por evaluacion."""
    r = r[r["estrato"] == estrato]
    filas = []
    for metrica in ("mcc", "auprc_minoritaria", "auroc"):
        p = r.pivot_table(index=["modelo", "semilla", "pliegue"], values=metrica).unstack("modelo")[metrica]
        for modelo in [m for m in ORDEN if m != MEJOR_INDIVIDUAL]:
            dif = (p[modelo] - p[MEJOR_INDIVIDUAL]).dropna()
            filas.append({"modelo": modelo, "metrica": metrica, "evaluaciones": len(dif),
                          "dif_media": dif.mean(), "dif_desv": dif.std(),
                          "mejor_en": int((dif > 0).sum())})
    c = pd.DataFrame(filas).set_index(["modelo", "metrica"])
    c["cumple"] = (c["dif_media"].gt(c.index.get_level_values("metrica").map(UMBRALES).astype(float))
                   & c["mejor_en"].ge(MINIMO_PLIEGUES))
    mejora = c["cumple"].unstack("metrica")[list(PRINCIPALES)].all(axis=1).rename("mejora")
    return c.round(4), mejora


comparacion_todos, mejora_todos = comparacion(res)
display(comparacion_todos.unstack("metrica")[["dif_media", "mejor_en"]].round(3))
mejora_todos.to_frame()

Umbrales, la desviacion de svm_rbf entre sus 15 evaluaciones: {'mcc': 0.073, 'auprc_minoritaria': 0.041}


dif_media                        mejor_en          
metrica          auprc_minoritaria  auroc    mcc auprc_minoritaria auroc mcc
modelo                                                                      
bagging_svm                  0.002  0.002 -0.007                11    11   4
boosting                     0.000  0.011  0.000                 8    12   8
boosting afinado             0.001  0.010 -0.002                 9    12   6
bosque                       0.011  0.014 -0.002                12    14   8
bosque afinado               0.009  0.014  0.001                10    15   9
knn                         -0.021 -0.005 -0.032                 2     7   2
logistica                   -0.012 -0.006 -0.015                 4     4   3
mlp                         -0.020 -0.008 -0.027                 2     3   3
stacking                     0.009  0.013  0.009                11    14  12
votacion                     0.011  0.014  0.008                14    15   9

,mejora
modelo,
bagging_svm,False
boosting,False
boosting afinado,False
bosque,False
bosque afinado,False
knn,False
logistica,False
mlp,False
stacking,False


### Si dos modelos se distinguen o no

La tabla ordena por la media, y eso no prueba que la primera fila se distinga de la segunda. Aquí se remuestrean **grupos** -no filas, porque la fila no es la unidad independiente- para el mejor individual y los dos primeros ensambles de la tabla. Si los intervalos se solapan, ninguno gana y eso es lo que hay que decir.

In [8]:
candidatos = [MEJOR_INDIVIDUAL] + [m for m in tabla.index if m in ENSAMBLES][:2]
filas = []
for m in candidatos:
    auroc, lo, hi = intervalo_por_grupos(y, fdp[m][0], g)   # remuestrea grupos, no filas
    filas.append({"modelo": m, "auroc": auroc, "inferior": lo, "superior": hi})
intervalos = pd.DataFrame(filas).set_index("modelo").round(4)
intervalos

,auroc,inferior,superior
modelo,,,
svm_rbf,0.9208,0.8963,0.9408
stacking,0.9312,0.9089,0.9505
votacion,0.9338,0.9114,0.9523


### Por origen (exploratorio, no decide)

Un promedio puede esconder cómo le va a los productos naturales, que son los que el proyecto quiere priorizar. El [03c](03c_Permeabilidad_huellas.ipynb) y el [04c](04c_Regresion_actividad_huellas.ipynb) encontraron que las huellas solo ayudaban en los compuestos sin equivalente natural; aquí se mira si los ensambles reparten su ganancia igual o distinto.

In [9]:
por_origen = pd.unique(res.loc[res["estrato"].ne("todos"), "estrato"])
mcc_origen = pd.concat({e: res[res["estrato"].eq(e)].groupby("modelo")["mcc"].mean().loc[ORDEN]
                        for e in por_origen}, axis=1).round(4)
display(mcc_origen)
pd.concat({e: comparacion(res, e)[0].query("metrica in @PRINCIPALES")[["dif_media", "mejor_en", "cumple"]]
           for e in por_origen}, names=["origen"])

,sin equivalente natural,natural exacto,análogo estereoquímico
modelo,,,
svm_rbf,0.4989,0.7182,0.6958
logistica,0.5001,0.6924,0.6916
mlp,0.5406,0.6703,0.6558
knn,0.4762,0.6935,0.6187
bosque,0.5218,0.7145,0.6660
boosting,0.5209,0.7190,0.6830
bagging_svm,0.4958,0.7092,0.6911
votacion,0.5367,0.7204,0.6900
stacking,0.5527,0.7194,0.6841


dif_media  \
origen                  modelo           metrica                        
sin equivalente natural logistica        mcc                   0.0012   
                        mlp              mcc                   0.0417   
                        knn              mcc                  -0.0228   
                        bosque           mcc                   0.0229   
                        boosting         mcc                   0.0220   
                        bagging_svm      mcc                  -0.0031   
                        votacion         mcc                   0.0378   
                        stacking         mcc                   0.0538   
                        bosque afinado   mcc                   0.0157   
                        boosting afinado mcc                   0.0215   
                        logistica        auprc_minoritaria    -0.0135   
                        mlp              auprc_minoritaria    -0.0006   
                        knn              auprc_minoritaria    -0.0258   
                        bosque           auprc_minoritaria     0.0166   
                        boosting         auprc_minoritaria     0.0039   
                        bagging_svm      auprc_minoritaria     0.0024   
                        votacion         auprc_minoritaria     0.0455   
                        stacking         auprc_minoritaria     0.0406   
                        bosque afinado   auprc_minoritaria     0.0140   
                        boosting afinado auprc_minoritaria     0.0096   
natural exacto          logistica        mcc                  -0.0258   
                        mlp              mcc                  -0.0479   
                        knn              mcc                  -0.0247   
                        bosque           mcc                  -0.0037   
                        boosting         mcc                   0.0008   
                        bagging_svm      mcc                  -0.0090   
                        votacion         mcc                   0.0022   
                        stacking         mcc                   0.0012   
                        bosque afinado   mcc                   0.0013   
                        boosting afinado mcc                  -0.0063   
                        logistica        auprc_minoritaria    -0.0156   
                        mlp              auprc_minoritaria    -0.0211   
                        knn              auprc_minoritaria    -0.0136   
                        bosque           auprc_minoritaria     0.0133   
                        boosting         auprc_minoritaria     0.0023   
                        bagging_svm      auprc_minoritaria     0.0019   
                        votacion         auprc_minoritaria     0.0057   
                        stacking         auprc_minoritaria     0.0051   
                        bosque afinado   auprc_minoritaria     0.0139   
                        boosting afinado auprc_minoritaria     0.0029   
análogo estereoquímico  logistica        mcc                  -0.0042   
                        mlp              mcc                  -0.0400   
                        knn              mcc                  -0.0771   
                        bosque           mcc                  -0.0298   
                        boosting         mcc                  -0.0128   
                        bagging_svm      mcc                  -0.0047   
                        votacion         mcc                  -0.0059   
                        stacking         mcc                  -0.0117   
                        bosque afinado   mcc                  -0.0174   
                        boosting afinado mcc                  -0.0075   
                        logistica        auprc_minoritaria     0.0070   
                        mlp              auprc_minoritaria    -0.0086   
                        knn              auprc_minoritaria     0.0035   
                        bosque           aupr

### La brecha entre entrenamiento y validación

Un modelo que se aprende los ejemplos acierta mucho en lo que vio y mucho menos en lo que no. La brecha es el MCC sobre una muestra de 3,000 filas de entrenamiento menos el de validación, con la semilla 0 y sus cinco pliegues. Los ensambles de árboles ya sobreajustaban en el 03, así que se compara cada modelo contra los demás sabiendo eso. Con el `knn` la muestra de entrenamiento incluye a cada punto como su propio vecino, así que su brecha sale inflada y solo se compara contra sí misma.

In [10]:
filas = []
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
for k, (ajuste, val) in enumerate(cv.split(X, y, g), 1):
    muestra = np.random.default_rng(k).choice(ajuste, min(MUESTRA_ENTRENAMIENTO, len(ajuste)),
                                              replace=False)
    for nombre, pipe in MODELOS.items():
        m = clone(pipe)
        if getattr(pipe, "necesita_grupos", False):
            m.fit(X[ajuste], y[ajuste], grupos=g[ajuste])
        else:
            m.fit(X[ajuste], y[ajuste])
        filas.append({"modelo": nombre, "pliegue": k,
                      "mcc_entrenamiento": _medir(y[muestra], m.predict_proba(X[muestra])[:, 1])["mcc"],
                      "mcc_validacion": _medir(y[val], m.predict_proba(X[val])[:, 1])["mcc"]})
brecha = pd.DataFrame(filas).groupby("modelo")[["mcc_entrenamiento", "mcc_validacion"]].mean()
brecha["brecha"] = brecha["mcc_entrenamiento"] - brecha["mcc_validacion"]
brecha.loc[ORDEN].round(4)

,mcc_entrenamiento,mcc_validacion,brecha
modelo,,,
svm_rbf,0.7843,0.6959,0.0884
logistica,0.7444,0.6811,0.0633
mlp,0.9256,0.6574,0.2683
knn,0.7787,0.6651,0.1136
bosque,0.9551,0.6989,0.2562
boosting,0.9722,0.7034,0.2688
bagging_svm,0.7921,0.6916,0.1004
votacion,0.8350,0.7069,0.1281
stacking,0.8308,0.7079,0.1229


In [11]:
# Registro de la comparativa en experimentos/ (decisión 1.5: qué modelo y con qué representación).
from src.tracking import Corrida

with Corrida("ensambles-permeabilidad", decision="1.5",
             params={"individuales": list(individuales), "ensambles": ENSAMBLES,
                     "base_heterogeneos": list(BASE), "mejor_individual": MEJOR_INDIVIDUAL,
                     "pliegues": 5, "representacion": "descriptores",
                     "metricas_principales": list(PRINCIPALES),
                     "metrica_tercera_lectura": "auroc",
                     "rejillas": {n: {k: list(map(str, v)) for k, v in r.items()}
                                  for n, r in REJILLAS.items()},
                     "pliegues_internos_busqueda": 3,
                     "regla_mejora": {"umbral_por_metrica": UMBRALES,
                                      "minimo_evaluaciones": MINIMO_PLIEGUES, "de": 15,
                                      "criterio": "las dos metricas principales a la vez",
                                      "umbral": "mayor desviacion del mejor individual entre sus 15 evaluaciones",
                                      "comparacion_decisiva": f"agregado contra {MEJOR_INDIVIDUAL}",
                                      "estratos": "exploratorios",
                                      "estado": "no pre-registrada; la misma del 04c y el 03c"},
                     "estratos": "origen"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "bbb_meta.parquet",
                    PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet"]) as c:
    c.metricas({"comparativa": tabla.to_dict(orient="index"),
                "comparacion_todos": {f"{m}/{k}": v for (m, k), v in
                                      comparacion_todos.to_dict(orient="index").items()},
                "intervalos": intervalos.to_dict(orient="index"),
                "brecha": brecha.to_dict(orient="index"),
                **{e: res[res["estrato"].eq(e)].groupby("modelo")[["mcc", "auprc_minoritaria", "auroc"]]
                   .mean().to_dict(orient="index") for e in por_origen}})

print(c.carpeta.name)

2026-10-29-artemio-1.5-ensambles-permeabilidad


In [12]:
# Las 15 evaluaciones por modelo y estrato. El cuaderno lo **escribe** y no lo lee: las
# conclusiones salen de las tablas de arriba. Va después del registro a propósito, porque
# escribirlo antes dejaría el árbol sucio al registrar.
SALIDA = DATOS / "05_ensambles.csv"
res.to_csv(SALIDA, index=False)
print(f"{SALIDA.relative_to(RAIZ)} · {len(res):,} filas")

notebooks/datos/05_ensambles.csv · 660 filas


## 4. El modelo final

*La elección se escribe aquí después de leer las tablas, y no la decide la primera fila: la rúbrica pide argumentos más allá de la métrica. Lo que hay que pesar, dicho antes de ver los resultados para que la decisión no se acomode a ellos: si los intervalos se solapan, cuánto cuesta cada ajuste, cuánto se puede explicar del modelo -la [C.2](../docs/decisiones.md#dc-2) pide un control interpretable al lado-, y cómo le va con los productos naturales, que es el estrato que el proyecto quiere priorizar.*

*Por completar por el equipo.*

In [13]:
FINAL = ""                                       # la clave del modelo elegido, de ORDEN
MOTIVO = ""                                      # por qué, en una línea, para la bitácora de la prueba
assert FINAL in MODELOS and MOTIVO, "escribe FINAL y MOTIVO antes de seguir"

AssertionError: escribe FINAL y MOTIVO antes de seguir

## 5. Los cuatro gráficos

Los cuatro se dibujan con las **predicciones fuera de pliegue** del modelo final -las que devuelve el arnés, una por compuesto, hecha por un modelo que no lo vio al ajustar- y no con un modelo reajustado sobre todo: un gráfico hecho sobre datos que el modelo vio describe su memoria y no su desempeño. Las funciones están en `src/evaluation/graficos.py` y **dibujan, no interpretan**: la interpretación de cada uno la escribe el equipo abajo.

In [14]:
p_fdp = fdp[FINAL][0]                            # semilla 0, una predicción por compuesto
fig, ejes = plt.subplots(2, 2, figsize=(11, 9))
curva_roc(y, p_fdp, ax=ejes[0, 0], etiqueta=FINAL)
ejes[0, 0].set_title("1. Curva ROC")
curva_precision_exhaustividad(y, p_fdp, ax=ejes[0, 1], etiqueta="BBB- (minoritaria)")
ejes[0, 1].set_title("2. Precision-exhaustividad de la minoritaria")
matriz_confusion(y, p_fdp, ax=ejes[1, 0], nombres=("BBB-", "BBB+"))
ejes[1, 0].set_title("3. Matriz de confusion")
plt.tight_layout()

KeyError: ''

**Interpretación de los tres primeros.** *Por completar por el equipo, mirando las figuras.*

### El cuarto: qué características sostienen la predicción

Dos métodos independientes, como exige la [C.2](../docs/decisiones.md#dc-2): la caída de desempeño al permutar una columna, medida sobre el pliegue de evaluación, y la importancia por impureza del bosque, que es barata y se sesga hacia variables de muchos valores. Se reporta **solo lo que los dos coinciden en poner arriba**, que es la parte defendible; la barra en azul fuerte marca esa intersección.

La permutación se mide sobre el modelo final y la impureza sobre el bosque, porque no todos los modelos tienen una importancia interna. Cualquier afirmación estructural que salga de aquí hay que contrastarla contra TPSA (< 67 Å²) y los donadores de hidrógeno (<= 1) antes de presentarla, porque esas dos propiedades dominan empíricamente la permeabilidad.

In [15]:
filas = []
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
bosque = clone(MODELOS["bosque"])
for k, (ajuste, val) in enumerate(cv.split(X, y, g), 1):
    m = clone(MODELOS[FINAL])
    if getattr(MODELOS[FINAL], "necesita_grupos", False):
        m.fit(X[ajuste], y[ajuste], grupos=g[ajuste])
    else:
        m.fit(X[ajuste], y[ajuste])
    perm = permutation_importance(m, X[val], y[val], n_repeats=5, random_state=k,
                                  scoring="matthews_corrcoef")
    imp = bosque.fit(X[ajuste], y[ajuste]).named_steps["modelo"].feature_importances_
    for j, nombre in enumerate(t.meta.attrs.get("columnas", range(X.shape[1]))):
        filas.append({"caracteristica": str(nombre), "pliegue": k,
                      "permutacion": perm.importances_mean[j], "impureza": imp[j]})
importancias = pd.DataFrame(filas)
fig, ax = plt.subplots(figsize=(7, 6))
importancias_dos_metodos(importancias, ax=ax)
ax.set_title("4. Importancia por permutacion, con la interseccion marcada")
plt.tight_layout()

KeyError: ''

**Interpretación del cuarto.** *Por completar por el equipo.*

### La prueba reservada, una sola vez

Hasta aquí todo se midió con validación cruzada sobre entrenamiento y validación. El objetivo 3.6 pide evaluar en datos no vistos, y esta es **la única vez** que el proyecto toca la partición de prueba: 1,499 compuestos que ningún cuaderno anterior leyó, con 57.3% BBB+ contra 64.0% del entrenamiento, porque los andamios raros se le asignaron a propósito ([1.2](../docs/decisiones.md#d1-2)).

El modelo final se reajusta con **todo** entrenamiento y validación -más datos, mismos hiperparámetros- y predice la prueba una vez. `tomar_prueba` exige `final=True` y un motivo, y escribe la fila en `experimentos/prueba.csv`, que va versionado: si alguien la tocara dos veces, se vería en el diff. La llamada va **dentro** de su propio `with Corrida(...)`, porque esa escritura ensucia el árbol y hacerla antes de entrar al contexto registraría `arbol_sucio: true` sin que nadie haya tocado el código.

Las cifras de la prueba **no se comparan** con las de la tabla de arriba como si fueran lo mismo: allí son promedios de 15 evaluaciones sobre una distribución parecida a la de entrenamiento, y aquí es una sola medición sobre una distribución distinta a propósito. Se espera que bajen.

In [16]:
# La prueba se mide dentro de su propia corrida, y `tomar_prueba` se llama **dentro** del
# contexto a propósito: escribe una línea en experimentos/prueba.csv, que va versionado, así
# que llamarla antes de entrar haría que la corrida registrara `arbol_sucio: true`.
meta = pd.read_parquet(PROCESADOS / "bbb_meta.parquet").assign(fila=lambda d: np.arange(len(d)))
completo = cargar("permeabilidad", estratos="origen", incluir_prueba=True)

with Corrida("modelo-final-permeabilidad", decision="1.5",
             params={"modelo_final": FINAL, "motivo": MOTIVO,
                     "representacion": "descriptores",
                     "ajustado_con": "entrenamiento y validacion, sin la prueba",
                     "prueba": "una sola vez, por tomar_prueba(final=True)",
                     "estratos": "origen"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "bbb_meta.parquet",
                    PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet"]) as c:
    prueba = tomar_prueba(meta.rename(columns={"origen": "estrato"}), final=True, motivo=MOTIVO)
    idx = prueba["fila"].to_numpy()
    modelo_final = clone(MODELOS[FINAL])
    if getattr(MODELOS[FINAL], "necesita_grupos", False):
        modelo_final.fit(X, y, grupos=g)
    else:
        modelo_final.fit(X, y)
    p_prueba = modelo_final.predict_proba(completo.X[idx])[:, 1]
    y_prueba = completo.y[idx]
    origen_prueba = completo.estratos[idx]
    en_prueba = {"todos": _medir(y_prueba, p_prueba)}
    for e in pd.unique(origen_prueba):
        sel = origen_prueba == e
        ys = y_prueba[sel]
        if min(int(ys.sum()), int((1 - ys).sum())) >= 20:    # el mismo mínimo que el arnés
            en_prueba[str(e)] = _medir(ys, p_prueba[sel])
    c.metricas({"prueba": en_prueba, "validacion_cruzada": tabla.loc[FINAL].to_dict()})

print(c.carpeta.name)

ValueError: Debes especificar un 'motivo' para evaluar el conjunto de prueba.

### Las cifras de la prueba, en total y por origen

In [17]:
pd.DataFrame(en_prueba).T.assign(
    n=[len(y_prueba)] + [int((origen_prueba == e).sum()) for e in list(en_prueba)[1:]],
    positivos=[y_prueba.mean()] + [y_prueba[origen_prueba == e].mean() for e in list(en_prueba)[1:]],
).round(4)

NameError: name 'en_prueba' is not defined

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Andamiaje del avance, escrito antes de ver cualquier resultado: los cinco ensambles de `src/models/ensambles_clasificacion.py`, la partición agrupada de la validación interna del stacking, el armado de la comparativa pareada y el esqueleto de este cuaderno | Secciones 1 a 3 y 5 (el código de apoyo vive en `src/`) y el esqueleto de las demás. El plan reserva al equipo, sin IA, cuáles ensambles y con qué bases, las rejillas, las lecturas, el argumento del modelo final y la interpretación de los cuatro gráficos | **Lo que el equipo decidió:** los cuatro mejores del 04 por MCC como base de la votación y del stacking, las dos rejillas por encima del mínimo que pide la rúbrica y las diez copias del bagging. **Lo que encontró la revisión:** que el módulo de ensambles colisionaba con el del 05b, que es de regresión, lo que llevó a separarlos por fase, y que a esta declaración le faltaban dos de sus tres partes. **Lo que se verificó:** `tests/test_ensambles_clasificacion.py` y `tests/test_05_codigo.py`, que ata el código mostrado al de `src/`, y una corrida de humo del cuaderno completo sobre una submuestra antes de ejecutarlo |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | *Por completar* |
| Artemio Santiago Padilla Robles | *Por completar* |
| José Antonio Vázquez Martínez | *Por completar* |

### Decisiones principales

**Decisión 1: cada ensamble se compara contra el mejor individual medido en esta misma corrida, y no contra la cifra registrada del Avance 4.**

- Justificación: la columna de tiempos solo significa lo mismo en todas las filas si salen de la misma máquina, las mismas semillas y los mismos pliegues; y la matriz de descriptores de la fase 1 no es idéntica entre las máquinas del equipo, porque la misma versión de RDKit calcula flotantes distintos según cómo se compiló ([#166](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/166)). Restar contra el 0.703 registrado mezclaría el efecto del ensamble con el de la matriz.
- Alternativa descartada y motivo: tomar ese 0.703 como referencia y ahorrarse cuatro ajustes por pliegue. Se descartó porque esperar a que se declare cuál matriz es la canónica bloquearía el avance, y la diferencia pareada dentro de una sola corrida no depende de esa decisión.

**Decisión 2: las bases de la votación y del stacking son los cuatro mejores del Avance 4 por MCC, no las seis familias.**

- Justificación: los cuatro que quedaron arriba son de familias distintas -margen, lineal, red y vecinos-, que es la diversidad que un ensamble heterogéneo necesita. Añadir las dos que quedaron abajo sube el costo de cada ajuste y no hay ninguna medición que haga esperar que suban el promedio.
- Alternativa descartada y motivo: las seis familias del 04. Lo que haría reconsiderarlo es que la votación mejore al SVM y el stacking no: eso apuntaría a que el promedio depende de qué bases entran, y entonces vale medir con las seis.

**Decisión 3: el stacking parte por dentro con los grupos del pliegue externo.**

- Justificación: su meta-modelo aprende de predicciones que las bases hacen fuera de pliegue, y `StackingClassifier` con un `cv` entero arma esos pliegues al azar. Con análogos del mismo esqueleto a los dos lados, esas predicciones salen mejores de lo que serán en validación y el meta-modelo aprende a confiar de más en el que memoriza; es la misma razón de la decisión [1.2](../docs/decisiones.md#d1-2).
- Alternativa descartada y motivo: el `cv=3` por omisión. Se descartó porque la fuga no aparecería en ninguna métrica: el ajuste externo seguiría viéndose honesto. Por eso `StackingAgrupado` revienta si no recibe los grupos, en vez de partir al azar en silencio.

### Responsabilidad

Las salidas de las herramientas se usaron como insumo de trabajo y fueron revisadas,
verificadas, editadas y adaptadas por el equipo. El equipo asume la responsabilidad
por la exactitud, la integridad y el contenido final del avance.
